In [87]:
#librerie di base
import pandas as pd
import numpy as np

In [88]:
# =============================================================
# =============================================================
#
# importo le funzioni personali creati per questa fase


#aggiunto quando ho spostato i file in una cartella a parte
import sys
sys.path.append("./Moduli")

import Moduli.Modulo_Common as pd_common

In [89]:
df_comuni = pd.read_csv( pd_common.GetFolderAnalisi()+ "//ElencoComuni.csv")

In [90]:
df_forecast_base = df_comuni.copy()

In [91]:
df_forecast_base.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 135649 entries, 0 to 135648
Data columns (total 7 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0        135649 non-null  int64  
 1   anno_riferimento  135649 non-null  int64  
 2   COD_UTS           135649 non-null  int64  
 3   PRO_COM           135649 non-null  int64  
 4   COMUNE            135649 non-null  object 
 5   AREA_KMQ          135649 non-null  float64
 6   POP_RES           135649 non-null  int64  
dtypes: float64(1), int64(5), object(1)
memory usage: 7.2+ MB


In [92]:
df_forecast_base = df_forecast_base.drop('COD_UTS',axis=1)

In [93]:
df_forecast_base = df_forecast_base.drop('COMUNE',axis=1)

In [94]:
#considero la situazione amministrativa dei comuni del 2026
df_forecast_base = df_forecast_base[df_forecast_base.anno_riferimento==2026]

In [95]:
df_forecast_base['anno_riferimento'].unique()

array([2026])

In [96]:
#POSSO RIMUOVERE IL CAMPO CHE SI RIFEREISCE ALL'UNICO anno
df_forecast_base = df_forecast_base.drop('anno_riferimento',axis=1)

In [97]:
#devo incrementare il dettaglio per avere l'univocità della densità
df_forecast_base['POP_AL_KMQ']= round(df_forecast_base['POP_RES'] / df_forecast_base['AREA_KMQ'] ,6)

In [98]:
#rimuovo l'informazione da cui derivo
df_forecast_base = df_forecast_base.drop('POP_RES',axis=1)

In [99]:
df_forecast_base = df_forecast_base.drop('AREA_KMQ',axis=1)

In [100]:
# creao il campo temporale di riferimento del forecast
#essendo una future qualsiasi, il fatto di passare dal 2024 al 2027 non dovrebbe 
# impattare tanto se non per l'incremento della distanza tra 2027 al 2024
df_forecast_base['TIME_PERIOD']= 2027

In [101]:
# il target ovviamente è da calcolare
df_forecast_base['OBS_VALUE']= 0

In [102]:
#aggiungo la feature legata ai Morti\Feriti
# inizio con i feriti
df_forecast_base['MORTI']= 0

In [103]:
# valore relativo ai feriti
df_forecast_base['RESULT']= 'F'
df_forecast_base['RESULT_DESC']= 'Feriti'

In [104]:
#riordino i dati per accodare i dataset
df_forecast_base= df_forecast_base[['TIME_PERIOD', 'PRO_COM', 'POP_AL_KMQ', 'RESULT','RESULT_DESC', 'OBS_VALUE', 'MORTI']]

In [105]:
df_forecast_base.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7895 entries, 127754 to 135648
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   TIME_PERIOD  7895 non-null   int64  
 1   PRO_COM      7895 non-null   int64  
 2   POP_AL_KMQ   7895 non-null   float64
 3   RESULT       7895 non-null   object 
 4   RESULT_DESC  7895 non-null   object 
 5   OBS_VALUE    7895 non-null   int64  
 6   MORTI        7895 non-null   int64  
dtypes: float64(1), int64(4), object(2)
memory usage: 493.4+ KB


In [106]:
df_forecast_base['RESULT'] = df_forecast_base['RESULT'].astype('string')
df_forecast_base['RESULT_DESC'] = df_forecast_base['RESULT_DESC'].astype('string')

In [107]:
df_forecast_base.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7895 entries, 127754 to 135648
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   TIME_PERIOD  7895 non-null   int64  
 1   PRO_COM      7895 non-null   int64  
 2   POP_AL_KMQ   7895 non-null   float64
 3   RESULT       7895 non-null   string 
 4   RESULT_DESC  7895 non-null   string 
 5   OBS_VALUE    7895 non-null   int64  
 6   MORTI        7895 non-null   int64  
dtypes: float64(1), int64(4), string(2)
memory usage: 493.4 KB


PREPARO CASO FERITI

In [108]:
#definisco il dataset corrente come quello appunto legato ai feriti
df_forecast_feriti = df_forecast_base.copy()

PREPARO CASO MORTI

In [109]:
#definisco il dataset invece relativo ai morti
df_forecast_morti = df_forecast_base.copy()

In [110]:
#aggiorno i dati perchè si riferisce ai morti
df_forecast_morti['MORTI']= 1

In [111]:
df_forecast_morti['RESULT']= 'M'
df_forecast_morti['RESULT_DESC']= 'Morti'

UNIONE DEI 2 CASI

In [112]:
# mi creo quindi il dataset del forecast per tutte le combinazioni necessarie
df_forecast = pd.concat([df_forecast_feriti, df_forecast_morti])

eseguo le verifiche sui conti dei dataset sorgenti e finali

In [113]:
rows_feriti, columns = df_forecast_feriti.shape
print(rows_feriti)

7895


In [114]:
rows_morti, columns = df_forecast_morti.shape
print(rows_morti)

7895


In [115]:
print(f'{str(rows_feriti + rows_morti)}')

15790


In [116]:
rows_forecast, columns = df_forecast.shape
print(rows_forecast)

15790


Aggiunta di informazioni da usare per l'aggiornamento dei dati previsionali calcolati dai modelli

In [117]:
df_forecast = df_forecast.reset_index()

In [118]:
df_forecast["row_num"] = range(len(df_forecast))

In [119]:
#assegno l'ambito dei dati forecast come appunto di forecast
df_forecast['TIPO_DATO'] ='F'
df_forecast['TIPO_DATO_DSC'] ='Forecast'

In [120]:
df_comuni_flat = pd.read_csv( pd_common.GetFolderAnalisi() + "//ElencoComuniProvinceRegioniRipGeografiche.csv")

In [121]:
df_comuni_flat.columns

Index(['Unnamed: 0', 'anno_riferimento', 'COD_RIP', 'DEN_RIP', 'COD_REG',
       'DEN_REG', 'COD_UTS', 'DEN_UTS', 'PRO_COM', 'COMUNE', 'AREA_KMQ',
       'POP_RES', 'POP_AL_KMQ'],
      dtype='object')

In [122]:
#ovviamente prendo l'anno di riferimento dei comuni che ho considerato
df_comuni_flat = df_comuni_flat[df_comuni_flat['anno_riferimento']==2026]

In [123]:
df_comuni_flat.anno_riferimento.unique()

array([2026])

In [124]:
df_forecast = df_forecast.merge(df_comuni_flat[['PRO_COM',
                                                'COD_RIP', 'DEN_RIP',
                                                'COD_REG', 'DEN_REG',
                                                'COD_UTS', 'DEN_UTS',
                                                'COMUNE',
                                                'AREA_KMQ','POP_RES']],
                                            how='inner',
                                            left_on='PRO_COM',
                                            right_on='PRO_COM')

In [125]:
df_forecast.columns

Index(['index', 'TIME_PERIOD', 'PRO_COM', 'POP_AL_KMQ', 'RESULT',
       'RESULT_DESC', 'OBS_VALUE', 'MORTI', 'row_num', 'TIPO_DATO',
       'TIPO_DATO_DSC', 'COD_RIP', 'DEN_RIP', 'COD_REG', 'DEN_REG', 'COD_UTS',
       'DEN_UTS', 'COMUNE', 'AREA_KMQ', 'POP_RES'],
      dtype='object')

SALVO I DATI AD USO DELLE ANALISI DI FORECAST

In [126]:
#ordino i dati come incidenti_flat
#le ultime colonne sono di supporto per join e per feature, da rimuovere in seguito
df_forecast = df_forecast[['index', 'TIME_PERIOD', 
            'COD_RIP', 'DEN_RIP',
            'COD_REG', 'DEN_REG',
            'COD_UTS', 'DEN_UTS',
            'PRO_COM', 'COMUNE',
            'AREA_KMQ', 'POP_RES','POP_AL_KMQ', 
            'RESULT', 'RESULT_DESC',
            'OBS_VALUE',
            'TIPO_DATO', 'TIPO_DATO_DSC',
            'MORTI',             
            'row_num'          
       ]]

In [127]:
#caso vs:
#   importare le librerie come negli altri file
pd_common.SalvaDataset(df_forecast, f"Forecast_Da_Calcolare",False)
#
#df_forecast.to_csv('.//Analisi//Forecast_Da_Calcolare.csv')

      path completo file:  c:\progetto_dataanalyst\progettofinale\Analisi/Forecast_Da_Calcolare.csv
         Il file 'c:\progetto_dataanalyst\progettofinale\Analisi/Forecast_Da_Calcolare.csv' esiste, per cui lo rimuovo
      Il file salvato c:\progetto_dataanalyst\progettofinale\Analisi/Forecast_Da_Calcolare.csv
